In [105]:
from typing import TypedDict
import importlib
from langgraph.graph import StateGraph, END,START
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import notes_ai
notes_ai = importlib.reload(notes_ai)
chercher_notes = notes_ai.chercher_notes
load_dotenv(dotenv_path="/home/awabzem/Documents/rt/pfa/.env")
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_mcp_adapters.client import MultiServerMCPClient
import asyncio
import nest_asyncio


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3476.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [106]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio

nest_asyncio.apply()

server_config = {
    "graph": {
        "url": "http://localhost:8000/mcp/",
        "transport": "sse"
    },
    
}

client = MultiServerMCPClient(server_config)

tools = await client.get_tools()

In [107]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio

nest_asyncio.apply()

server_config = {
    "graph": {
        "url": "http://localhost:8000/mcp/",
        "transport": "sse"
    },
    "vector": {
        "url": "http://localhost:8005/mcp/",
        "transport": "http"
    }
}

client = MultiServerMCPClient(server_config)

tools = await client.get_tools()

In [108]:
from langgraph.graph import StateGraph, END,START

# 1. Define state
from typing import List, TypedDict


class State(TypedDict):
    user_query: str
    search_queries: List[str]
    notes: List[str]
    answer: str

In [109]:
llm = ChatGroq(
    model="llama-3.1-8b-instant", 
      # fast + good for testing
)
checkpointer = InMemorySaver()


In [110]:
def planner_node(state):
    question = state["user_query"]

    prompt = f"""
You are a search planner.

Convert the user question short search queries.

Rules:
- Output ONLY a Python list of strings
- No explanation
- No numbering
- Queries must be short and keyword-based

User question:
{question}
"""

    response = llm.invoke(prompt)

    # convert string output → python list safely
    queries = eval(response.content)

    return {
        "search_queries": queries
    }

In [111]:
print(tools)

[StructuredTool(name='get_neo4j_schema', description="Returns nodes, their properties (with types and indexed flags), and relationships\nusing APOC's schema inspection.\n\nYou should only provide a `sample_size` value if requested by the user, or tuning the retrieval performance.\n\nPerformance Notes:\n    - If `sample_size` is not provided, uses the server's default sample setting defined in the server configuration.\n    - If retrieving the schema times out, try lowering the sample size, e.g. `sample_size=100`.\n    - To sample the entire graph use `sample_size=-1`.", args_schema={'properties': {'sample_size': {'default': None, 'description': 'The sample size used to infer the graph schema. Larger samples are slower, but more accurate. Smaller samples are faster, but might miss information.', 'type': 'integer'}}, 'type': 'object'}, metadata={'title': 'Get Neo4j Schema', 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': True, '_meta': {'_fastmcp'

In [112]:
async def retriever_node(state, tools):
    all_notes = []

    search_tool = next(t for t in tools if t.name == "search_notes")

    for q in state["search_queries"]:
        response = await search_tool.ainvoke({
            "query": q,
            "k": 3
        })

        if isinstance(response, dict):
            notes = response.get("results", [])
        elif isinstance(response, list):
            notes = response
        else:
            notes = []

        for note in notes:
            if isinstance(note, dict):
                normalized = (
                    note.get("note")
                    or note.get("content")
                    or note.get("text")
                    or note.get("document")
                    or str(note)
                )
            else:
                normalized = str(note)

            if normalized:
                all_notes.append(normalized)

    # remove duplicates while keeping order
    all_notes = list(dict.fromkeys(all_notes))

    return {
        "notes": all_notes
    }

In [113]:
def answer_node(state):
    question = state["user_query"]
    notes = state["notes"]

    context = "\n".join([f"- {n}" for n in notes])

    prompt = f"""
You are a second brain assistant.

You MUST answer ONLY using the provided notes.
If the notes do not contain the answer, say: "I don't have enough information in my notes."

---

NOTES:

{context}

---

QUESTION:
{question}

---

Answer clearly and concisely:
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [114]:
from langgraph.prebuilt import ToolNode


tool_node= ToolNode(tools)

In [115]:
from functools import partial

builder = StateGraph(State)
builder.add_node("planner", planner_node)
builder.add_node("retriever", partial(retriever_node, tools=tools))
builder.add_node("answer", answer_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "answer")
builder.add_edge("answer", END)

graph = builder.compile(checkpointer=checkpointer)

In [116]:
config = {"configurable": {"thread_id": "1"}}
result = await graph.ainvoke(
    {"user_query": "Explain AI in 1 sentence"},
    config=config
)

In [117]:
print(result)

{'user_query': 'Explain AI in 1 sentence', 'search_queries': ['What is AI', 'AI definition', 'AI explanation'], 'notes': ['{"query":"What is AI","results":["hello mcp"]}', '{"query":"AI definition","results":["hello mcp"]}', '{"query":"AI explanation","results":["hello mcp"]}'], 'answer': 'According to my notes, AI can be explained in one sentence as "hello mcp."'}


In [118]:
from langsmith import Client

client = Client()
dataset_name = "Notes_App_Test_Dataset"

# Create a new dataset in LangSmith
datasets = client.list_datasets()

dataset = next((d for d in datasets if d.name == dataset_name), None)

# Add some test examples (Input = user_query, Output = expected_answer)
examples =[
(
        {"user_query": "How do I undo my last git commit without losing changes?"}, 
        {"expected_answer": "You can use the command 'git reset --soft HEAD~1'."}
    ),
    (
        {"user_query": "What should I get my Mom for her birthday?"}, 
        {"expected_answer": "You should get her a Kindle Paperwhite or a new set of ceramic gardening pots."}
    ),
    (
        {"user_query": "What is the ratio and temperature for a V60 pour over?"}, 
        {"expected_answer": "Use 15g of coffee and 250ml of water at 93°C."}
    ),
    (
        {"user_query": "What did I discuss with my manager in our 1-on-1?"}, 
        {"expected_answer": "You discussed your promotion timeline, improving cross-functional communication, and leading a major backend project this quarter."}
    )
]

for input_dict, output_dict in examples:
    client.create_example(
        inputs=input_dict,
        outputs=output_dict,
        dataset_id=dataset.id,
    )
print(f"Dataset '{dataset_name}' created successfully!")

Dataset 'Notes_App_Test_Dataset' created successfully!


In [119]:
def predict_answer(inputs: dict):
    # Extract the question from the dataset input
    question = inputs["user_query"]
    
    # Run your LangGraph
    config = {"configurable": {"thread_id": "eval_thread"}}
    result = graph.invoke({"user_query": question}, config=config)
    
    # Return ONLY the final answer for evaluation
    return {"answer": result["answer"]}

In [120]:
def correctness_evaluator(run, example):
    # Get the expected answer from the dataset
    expected = example.outputs["expected_answer"]
    # Get the actual answer from your graph
    actual = run.outputs["answer"]
    
    # Ask Groq to grade it, allowing it to think first
    prompt = f"""
    You are an expert grader evaluating an AI assistant.
    
    Expected Answer: {expected}
    Actual Answer: {actual}
    
    Task:
    1. Briefly explain if the Actual Answer contains the core factual information from the Expected Answer.
    2. It is okay if the Actual Answer provides extra context, as long as the core facts match.
    3. End your response with exactly: [GRADE: YES] or [GRADE: NO]
    """
    
    response = llm.invoke(prompt).content.strip().upper()
    
    # Safely parse the grade
    if "[GRADE: YES]" in response:
        score = 1
    elif "[GRADE: NO]" in response:
        score = 0
    else:
        # Fallback just in case the LLM formats it weirdly
        score = 0 
        
    return {
        "key": "correctness", 
        "score": score, 
        "comment": response # Optional: LangSmith will save the LLM's explanation!
    }

In [121]:
from langsmith.evaluation import evaluate

experiment_results = evaluate(
    predict_answer, # Your LangGraph wrapper
    data=dataset_name, # The dataset we created
    evaluators=[correctness_evaluator], # The grading function
    experiment_prefix="groq-llama-3-test", # Name of this test run
    client=client
)

View the evaluation results for experiment: 'groq-llama-3-test-43b11a34' at:
https://eu.smith.langchain.com/o/3a5ee616-d418-413e-bb48-8d292f03604d/datasets/0b919c23-aa11-40f3-a9ff-cd0aae1f29b0/compare?selectedSessions=138c7143-cc94-4f0f-9df8-d56df39f04e7




0it [00:00, ?it/s]Error running target function: No synchronous function provided to "retriever".
Either initialize with a synchronous function or invoke via the async API (ainvoke, astream, etc.)
Traceback (most recent call last):
  File "/home/awabzem/Documents/rt/pfa/.venv/lib/python3.13/site-packages/langsmith/evaluation/_runner.py", line 1939, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_893246/3177543282.py", line 7, in predict_answer
    result = graph.invoke({"user_query": question}, config=config)
  File "/home/awabzem/Documents/rt/pfa/.venv/lib/python3.13/site-packages/langgraph/pregel/main.py", line 3309, in invoke
    for chunk in self.stream(
                 ~~~~~~~~~~~^
        input,
        ^^^^^^
    ...<10 lines>...
        **kwargs,
        ^^^^^^^^^
    ):
    ^
  File "/home/awabzem/Documents/rt/pfa/.venv/lib/python3.13/site-packages/langgraph/pregel/main.py", line 2736, in stream


In [122]:
result = await graph.ainvoke(
    {"user_query": "What did I discuss with my manager in our 1-on-1?"},
    config=config
)

In [123]:
print(result)

{'user_query': 'What did I discuss with my manager in our 1-on-1?', 'search_queries': ['Manager 1-on-1 discussion', '1-on-1 with manager', 'Manager discussion notes'], 'notes': ['{"query":"Manager 1-on-1 discussion","results":["hello mcp"]}', '{"query":"1-on-1 with manager","results":["hello mcp"]}', '{"query":"Manager discussion notes","results":["hello mcp"]}'], 'answer': "I don't have enough information in my notes."}
